# n-gram Language Models — counting your way to P(next word)

> Tutorial pair for [`ngram_lm.py`](ngram_lm.py).

## 1. Intuition
A language model answers one question: *given the words so far, what comes
next?* The oldest, simplest answer is to **count**. If "the cat sat on the" was
usually followed by "mat" in our corpus, predict "mat". An n-gram model only
looks at the last $n-1$ words (a Markov shortcut), tallies what followed them,
and turns those tallies into probabilities. The whole art is what to do about
word sequences you have **never seen** — that is what *smoothing* fixes.

## 2. Concept (the slide)
- **n-gram:** a contiguous run of $n$ tokens. Bigram $n=2$, trigram $n=3$.
- **Markov assumption:** condition only on the previous $n-1$ tokens.
- **MLE:** estimate $P(w\mid h)$ as a normalized count.
- **Zero-count problem:** any unseen n-gram gets probability 0, sending
  perplexity to $\infty$. So we **smooth**:
  - *Add-k / Laplace*: pretend every n-gram was seen $k$ extra times.
  - *Stupid backoff*: if the n-gram is unseen, fall back to a shorter one,
    scaled by a constant $\alpha$ (cheap, unnormalized).
  - *Kneser-Ney*: discount a fixed mass $D$ and redistribute it using a
    **continuation** probability (how many distinct contexts a word completes).
- **Perplexity:** the geometric-mean branching factor — lower is better.

## 3. Math derivation

**Chain rule (exact).** Every sequence factorizes as
$$P(w_1,\dots,w_T)=\prod_{t=1}^{T} P(w_t\mid w_1,\dots,w_{t-1}).$$

**n-gram (Markov) approximation.** Truncate the history to the last $n-1$ tokens:
$$P(w_t\mid w_1^{t-1})\approx P(w_t\mid w_{t-n+1}^{t-1}).$$

**Maximum-likelihood estimate.** With $c(\cdot)$ the corpus count and history
$h=w_{t-n+1}^{t-1}$,
$$P_{\mathrm{MLE}}(w\mid h)=\frac{c(h,w)}{c(h)}=\frac{c(h,w)}{\sum_{w'}c(h,w')}.$$

**Add-k smoothing.** Add a pseudo-count $k$ to every word in the vocabulary $V$:
$$P_{+k}(w\mid h)=\frac{c(h,w)+k}{c(h)+k\,|V|}.$$
$k=1$ is *Laplace*. Larger $k$ moves mass toward the uniform distribution.

**Stupid backoff** (Brants et al., 2007) is a recursive, *unnormalized* score:
$$S(w\mid h)=\begin{cases}\dfrac{c(h,w)}{c(h)} & c(h,w)>0\\[1.2em]
\alpha\, S(w\mid h') & \text{otherwise,}\end{cases}$$
where $h'$ drops the oldest word of $h$. Fast and surprisingly strong at scale.

**Kneser-Ney (interpolated).** Subtract a fixed discount $D$ from each seen count
and add an interpolation with the lower order:
$$P_{KN}(w\mid h)=\frac{\max(c(h,w)-D,\,0)}{c(h)}+\lambda(h)\,P_{KN}(w\mid h'),$$
where the back-off weight is exactly the discounted mass we removed:
$$\lambda(h)=\frac{D\,\big|\{w:c(h,w)>0\}\big|}{c(h)}.$$
The **key idea** is the lowest-order term. Instead of the unigram frequency, KN
uses the **continuation probability** — how many *distinct* words $w$ follows:
$$P_{\mathrm{cont}}(w)=\frac{N_{1+}(\bullet,w)}{N_{1+}(\bullet,\bullet)},\qquad
N_{1+}(\bullet,w)=\big|\{v:c(v,w)>0\}\big|.$$
"Francisco" is frequent but almost only after "San", so $N_{1+}(\bullet,w)$ is
tiny — KN rightly gives it low probability in a novel context.

**Perplexity.** The model's per-token uncertainty, the exponentiated average
negative log-likelihood:
$$\mathrm{PP}(W)=\exp\!\Big(-\frac1N\sum_{i=1}^{N}\log P(w_i\mid h_i)\Big)
=\Big(\prod_i P(w_i\mid h_i)\Big)^{-1/N}.$$
A perplexity of $K$ means the model is as confused as if choosing uniformly among
$K$ words at each step.

## 4. NumPy implementation

In [ ]:
# ===== actual implementation from ngram_lm.py =====
from __future__ import annotations

import math

from collections import Counter, defaultdict

import numpy as np

SEED = 0

BOS = "<s>"

EOS = "</s>"

def _tokenize(text: str) -> list[str]:
    """Lowercase, keep words; split sentences on . ! ?"""
    import re
    sents = re.split(r"[.!?]+", text.lower())
    return [s.split() for s in sents if s.split()]

class NgramLM:
    r"""
    Count-based n-gram language model.

    **Chain rule.** Any sequence factorizes exactly as
        P(w_1..w_T) = \prod_t P(w_t | w_1..w_{t-1}).
    The **n-gram (Markov) assumption** truncates the history to the last n-1
    tokens:
        P(w_t | w_1..w_{t-1}) ≈ P(w_t | w_{t-n+1}..w_{t-1}).

    **MLE estimate** is just normalized counts:
        P(w | h) = count(h, w) / count(h).
    Unseen (h, w) -> 0, so we smooth. Modes:
      - "addk": (count(h,w)+k) / (count(h)+k|V|)
      - "backoff": stupid backoff, recursively shorten h with factor alpha
      - "kn": interpolated Kneser-Ney (absolute discount D + continuation prob)
    """

    def __init__(self, n=3, mode="kn", k=1.0, discount=0.75, alpha=0.4, seed=SEED):
        self.n = n
        self.mode = mode
        self.k = k
        self.D = discount
        self.alpha = alpha
        self.seed = seed

    # -- fitting: just collect counts of every order up to n -----------------
    def fit(self, sentences: list[list[str]]):
        self.vocab = {BOS, EOS}
        for s in sentences:
            self.vocab.update(s)
        self.V = len(self.vocab)
        # ngrams[m] maps a (context tuple of length m-1) -> Counter(next token)
        self.ngrams = [defaultdict(Counter) for _ in range(self.n + 1)]
        # Kneser-Ney continuation stats are defined at the BIGRAM level:
        #   N1+(.,w) = number of distinct words that precede w (as a bigram),
        #   N1+(.,.) = number of distinct bigram types in the corpus.
        self.cont_before = defaultdict(set)  # w -> {preceding words}
        bigram_types = set()
        for s in sentences:
            padded = [BOS] * (self.n - 1) + s + [EOS]
            for m in range(1, self.n + 1):
                for i in range(len(padded) - m + 1):
                    gram = tuple(padded[i:i + m])
                    ctx, w = gram[:-1], gram[-1]
                    self.ngrams[m][ctx][w] += 1
            # bigram-level continuation counts (independent of model order n)
            bpad = [BOS] + s + [EOS]
            for a, b in zip(bpad, bpad[1:]):
                self.cont_before[b].add(a)
                bigram_types.add((a, b))
        self.total_cont = max(len(bigram_types), 1)  # N1+(.,.)
        return self

    # -- probability of one token given its (already-truncated) history ------
    def prob(self, word: str, context: tuple) -> float:
        context = tuple(context)[-(self.n - 1):] if self.n > 1 else ()
        if self.mode == "addk":
            return self._prob_addk(word, context)
        if self.mode == "backoff":
            return self._prob_backoff(word, context)
        if self.mode == "kn":
            return self._prob_kn(word, context, self.n)
        raise ValueError(self.mode)

    def _prob_addk(self, word, context):
        # P(w|h) = (c(h,w)+k) / (c(h)+k|V|)
        counter = self.ngrams[len(context) + 1].get(context, Counter())
        num = counter.get(word, 0) + self.k
        den = sum(counter.values()) + self.k * self.V
        return num / den

    def _prob_backoff(self, word, context):
        # Stupid backoff (Brants 2007): use MLE if seen, else alpha * back off.
        # NOT a true distribution (doesn't sum to 1) but excellent in practice.
        counter = self.ngrams[len(context) + 1].get(context, Counter())
        c_hw = counter.get(word, 0)
        if c_hw > 0:
            return c_hw / sum(counter.values())
        if not context:
            # unigram floor so we never return exactly 0
            uni = self.ngrams[1][()]
            return self.alpha * (uni.get(word, 0) + 1) / (sum(uni.values()) + self.V)
        return self.alpha * self._prob_backoff(word, context[1:])

    def _prob_kn(self, word, context, order):
        r"""
        Interpolated Kneser-Ney with a single discount D:

            P_KN(w|h) = max(c(h,w)-D, 0)/c(h)  +  lambda(h) * P_KN(w|h')

        where lambda(h) = D * N1+(h, .) / c(h) is the leftover mass, and the
        lowest-order term uses the **continuation probability**
            P_cont(w) = N1+(. , w) / N1+(. , .)
        i.e. how many *distinct* contexts w follows, not how often it occurs.
        That is the Kneser-Ney insight: "Francisco" is frequent but only after
        "San", so its continuation probability is low.
        """
        if order == 1:
            # continuation probability + tiny floor so unseen words aren't 0
            n_w = len(self.cont_before.get(word, ()))
            return (n_w + 1.0) / (self.total_cont + self.V)
        counter = self.ngrams[order].get(context, Counter())
        c_h = sum(counter.values())
        if c_h == 0:
            return self._prob_kn(word, context[1:], order - 1)
        c_hw = counter.get(word, 0)
        first = max(c_hw - self.D, 0.0) / c_h
        n1 = len(counter)  # number of distinct words seen after this context
        lam = self.D * n1 / c_h
        return first + lam * self._prob_kn(word, context[1:], order - 1)

    # -- sentence / corpus log-probability and perplexity --------------------
    def log_prob_sentence(self, s: list[str]) -> tuple[float, int]:
        padded = [BOS] * (self.n - 1) + s + [EOS]
        logp, count = 0.0, 0
        for i in range(self.n - 1, len(padded)):
            ctx = tuple(padded[i - self.n + 1:i])
            p = self.prob(padded[i], ctx)
            logp += math.log(max(p, 1e-12))
            count += 1
        return logp, count

    def perplexity(self, sentences: list[list[str]]) -> float:
        # PP = exp(-(1/N) * sum log P(w_i | history))
        total_lp, total_n = 0.0, 0
        for s in sentences:
            lp, n = self.log_prob_sentence(s)
            total_lp += lp
            total_n += n
        return math.exp(-total_lp / max(total_n, 1))

    # -- generation: sample next token from the smoothed distribution --------
    def generate(self, max_len=20) -> list[str]:
        rng = np.random.default_rng(self.seed)
        vocab = sorted(self.vocab - {BOS})
        history = [BOS] * (self.n - 1)
        out = []
        for _ in range(max_len):
            ctx = tuple(history[-(self.n - 1):]) if self.n > 1 else ()
            probs = np.array([self.prob(w, ctx) for w in vocab], float)
            probs = probs / probs.sum()  # renormalize (backoff isn't normalized)
            w = vocab[rng.choice(len(vocab), p=probs)]
            if w == EOS:
                break
            out.append(w)
            history.append(w)
        return out

## 5. Reference — comparing smoothing methods (no neural net needed)

In [ ]:
# ===== actual implementation from ngram_lm.py =====
def compare_smoothing(train, test, n=3):
    """Return {mode: perplexity} on held-out text for several smoothers."""
    results = {}
    for mode in ("addk", "backoff", "kn"):
        lm = NgramLM(n=n, mode=mode).fit(train)
        results[mode] = lm.perplexity(test)
    return results

def toy_corpus() -> str:
    return (
        "the cat sat on the mat. "
        "the dog sat on the log. "
        "the cat chased the dog. "
        "the dog chased the cat around the mat. "
        "a happy cat naps on the warm mat. "
        "a happy dog runs around the green park. "
        "the cat and the dog are friends. "
        "the friendly dog naps on the log. "
        "cats and dogs sat on the mat together. "
        "the warm mat is where the cat naps."
    )

def demo():
    np.random.seed(SEED)
    sents = _tokenize(toy_corpus())
    train, test = sents[:8], sents[8:]

    print("=== Perplexity by smoothing method (trigram) ===")
    for mode, pp in compare_smoothing(train, test, n=3).items():
        print(f"  {mode:8s} test perplexity = {pp:7.3f}")

    print("\n=== Effect of add-k (k) on a bigram LM ===")
    for k in (0.01, 0.1, 1.0):
        lm = NgramLM(n=2, mode="addk", k=k).fit(train)
        print(f"  k={k:<4}  perplexity = {lm.perplexity(test):7.3f}")

    print("\n=== Generated sentences (Kneser-Ney trigram) ===")
    lm = NgramLM(n=3, mode="kn").fit(sents)
    for s in range(3):
        lm.seed = s
        print("  " + " ".join(lm.generate(max_len=12)))

    # sanity: probabilities given a context are a valid distribution (KN)
    lm = NgramLM(n=2, mode="kn").fit(sents)
    vocab = sorted(lm.vocab - {BOS})
    total = sum(lm.prob(w, ("the",)) for w in vocab)
    print(f"\nKN bigram P(.|'the') sums to {total:.4f} (should be ~1)")

## 6. Train / run — perplexity by smoother, the effect of k, and sampling

In [ ]:
demo()

## 7. Visualization — how add-k strength trades off perplexity

In [ ]:
import matplotlib
matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import ngram_lm as M

sents = M._tokenize(M.toy_corpus())
train, test = sents[:8], sents[8:]
ks = np.logspace(-3, 1, 25)

plt.figure(figsize=(7, 4))
for n, style in [(2, "-o"), (3, "-s")]:
    pps = []
    for k in ks:
        lm = M.NgramLM(n=n, mode="addk", k=float(k)).fit(train)
        pps.append(lm.perplexity(test))
    plt.plot(ks, pps, style, ms=4, label=f"{n}-gram add-k")
plt.xscale("log"); plt.xlabel("add-k pseudo-count k"); plt.ylabel("held-out perplexity")
plt.title("Too little smoothing overfits; too much -> uniform. There is a sweet spot.")
plt.legend(); plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Always smooth.** A single unseen n-gram gives a test sentence probability 0
  and perplexity $\infty$.
- **Add-k** is easy but blunt; with a big vocabulary it steals far too much mass.
  **Kneser-Ney** is the count-based gold standard because of the continuation
  trick — context diversity, not raw frequency, drives the back-off.
- **Sparsity explodes with $n$.** Higher-order models capture more context but
  almost everything becomes unseen; you lean entirely on back-off.
- Count models can't generalize across *similar* words ("dog" vs "puppy"). That
  is exactly what neural LMs with embeddings fix — see
  [`neural_lm.ipynb`](neural_lm.ipynb).
- Compare perplexities only with the **same vocabulary and tokenization**.